# Data cleaning for future values of features

This file applies the data cleaning pipeline to future values of the features (climate, population size, and upcoming datacenters). It is similar to the data cleaning pipeline for current and historical data, except that it does not involve river flow data (since that is the response we want to predict), and there are some additional data cleaning steps for upcoming data centers.  The final output is a file with monthly future values of features, which will be used to predict future values of river flow rates. 

In [48]:
#loading packages
import pandas as pd
import numpy as np
import re

from datetime import datetime, timedelta

### Climate Data

Loading climate data 

In [49]:
#loading data
future_climate = pd.read_csv("../Data/FormattedData/FutureClimateProjections.csv")

#converting time column to datetime format
future_climate["Time"] = pd.to_datetime(future_climate.Time)

future_climate.head()

,Time,Temp_Salt_SSP245,Temp_Verde_SSP245,Temp_Maricopa_SSP245,Precip_Salt_SSP585,Precip_Verde_SSP585,Precip_Maricopa_SSP585
0,2026-06-16 00:00:00,20.972582,21.974016,31.236763,6192.295867,3543.703456,834.732363
1,2026-07-16 12:00:00,25.034290,27.007235,36.358704,47477.990319,31365.076755,27081.072271
2,2026-08-16 12:00:00,20.949894,22.427137,32.220314,39420.638549,32705.906363,25530.815069
3,2026-09-16 00:00:00,18.813648,20.112804,30.477566,20961.371964,35531.386504,20287.838891
4,2026-10-16 12:00:00,15.049019,16.018923,25.397923,3136.122729,2153.111518,213.116947


Climate data is already at monthly scale but the timestamps are for the middle of each month instead of the first day. We need to fix that

In [50]:
future_climate["Time"]= future_climate["Time"].dt.to_period("M").dt.to_timestamp()
future_climate.head()

,Time,Temp_Salt_SSP245,Temp_Verde_SSP245,Temp_Maricopa_SSP245,Precip_Salt_SSP585,Precip_Verde_SSP585,Precip_Maricopa_SSP585
0,2026-06-01,20.972582,21.974016,31.236763,6192.295867,3543.703456,834.732363
1,2026-07-01,25.034290,27.007235,36.358704,47477.990319,31365.076755,27081.072271
2,2026-08-01,20.949894,22.427137,32.220314,39420.638549,32705.906363,25530.815069
3,2026-09-01,18.813648,20.112804,30.477566,20961.371964,35531.386504,20287.838891
4,2026-10-01,15.049019,16.018923,25.397923,3136.122729,2153.111518,213.116947


### Farmland area/irrigated farmland area in Maricopa county

Future projections for irrigated farmland area were not available, so we will just use the last observed value and assume that it stays constant in the near future.

In [51]:
#loading relevant columns of the formatted data file that contains all historical data
irrigated_acres = pd.read_csv("../Data/FormattedData/MonthlyFlow_and_AllFeatures.csv",
                              usecols=["Time", "IrrigatedLand_Acres"] )

#removing NAs
irrigated_acres.dropna(inplace=True)

#converting the Time column to datetime format
irrigated_acres["Time"] = pd.to_datetime(irrigated_acres.Time)

irrigated_acres.tail()

,Time,IrrigatedLand_Acres
1464,2026-01-01,204522.0
1465,2026-02-01,204522.0
1466,2026-03-01,204522.0
1467,2026-04-01,204522.0
1468,2026-05-01,204522.0


We will take the last observed value and duplicate it over the same time range as in the future climate data

In [52]:
#arranging the historic farmland data in descending order (latest data on top)
irrigated_acres = irrigated_acres.sort_values("Time", ascending=False)

#selecting the last observed value
last_observed_irrigated_acres = irrigated_acres.IrrigatedLand_Acres.iloc[0]

#creating a data frame with the same time range as the future climate dataframe
# and the last observed value of irrigated acres
future_irrigated_acres = pd.DataFrame({
    "Time":future_climate.Time,
    "IrrigatedLand_Acres":last_observed_irrigated_acres
})

future_irrigated_acres.head()

,Time,IrrigatedLand_Acres
0,2026-06-01,204522.0
1,2026-07-01,204522.0
2,2026-08-01,204522.0
3,2026-09-01,204522.0
4,2026-10-01,204522.0


### Population data

Loading future population data

In [53]:
future_population = pd.read_excel("../Data/RawData/Population/PopulationProjections_Maricopa.xlsx",
                                  sheet_name="1. Total Pop & Components",
                                  skiprows=3, skipfooter=8)

future_population.head()

,Year,Population,Population Change,Population % Change,Births,Deaths,Natural Change1,Net Domestic Migration,Net Foreign Migration,Total Net Migration2
0,2025,4.787790e+06,NaN,NaN,50945.000000,38298.000000,12647.000000,30449.000000,18817.169035,49266.169035
1,2026,4.882088e+06,94298.083483,0.019696,52308.341075,39556.766595,12751.574481,64973.360125,16573.148877,81546.509002
2,2027,4.973848e+06,91760.387871,0.018795,53617.069532,41150.173955,12466.895577,64580.360125,14713.132169,79293.492294
3,2028,5.063616e+06,89767.309871,0.018048,54881.297919,42793.469382,12087.828537,64186.360125,13493.121209,77679.481334
4,2029,5.152639e+06,89022.756898,0.017581,56112.800436,44529.526246,11583.274190,63793.360125,13646.122584,77439.482709


Linearly interpolating to monthly timescale

In [54]:
#converting Year column to a datetime format
future_population["Time"] = pd.to_datetime(future_population["Year"].astype(str) + "-01-01")
future_population = future_population.sort_values("Time") #sorting by time from oldest to latest

#indexing by time column
future_population.set_index("Time", inplace=True, drop=False)

#creating a monthly date index
monthly_index = pd.date_range(
    start="2026-01-01",
    end=future_climate.Time.max(),
    freq="MS"      # Month Start
)

#eeindexing and interpolating
futurepopulation_monthly = (
    future_population[["Population"]]
    .reindex(monthly_index)
    .interpolate(method="time"))

#adding a time column
futurepopulation_monthly["Time"] = futurepopulation_monthly.index

#filtering to keep only months after May 2026
futurepopulation_monthly = futurepopulation_monthly[futurepopulation_monthly.Time>"2026-05-01"]

futurepopulation_monthly.head()

,Population,Time
2026-06-01,4.920049e+06,2026-06-01
2026-07-01,4.927591e+06,2026-07-01
2026-08-01,4.935385e+06,2026-08-01
2026-09-01,4.943178e+06,2026-09-01
2026-10-01,4.950720e+06,2026-10-01


### Data centers

Loading file

In [55]:
#Loading  the data center file
upcoming_datacenters = pd.read_csv('../Data/RawData/phoenix_upcoming_datacenters.csv')

upcoming_datacenters.head()

,Project_Name,Developer,Capacity_MW,Expected_Operational_Year,Status,Notes
0,5C Data Centers PHX01,5C Data Centers,20.0,2026,NaN,NaN
1,EdgeConneX PHX11,EdgeConneX,115.0,Not specified,under construction,NaN
2,Fortescue Buckeye Data Center,Fortescue,NaN,Not specified,zoning and permitting,NaN
3,Hassayampa Ranch Data Center,Arizona Land Consulting,1500.0,Not specified,zoning and permitting,NaN
4,Iron Mountain AZP-3,Iron Mountain,36.0,2026,NaN,NaN


In [56]:
upcoming_datacenters.Expected_Operational_Year.unique()

<ArrowStringArray>
['2026', 'Not specified', '2028', '2027', '2039']
Length: 5, dtype: str

In [57]:
upcoming_datacenters.Status.unique()

<ArrowStringArray>
[nan, 'under construction', 'zoning and permitting', 'pre-construction']
Length: 4, dtype: str

For several upcoming data centers, the year they are expected to become operational is not known. We will estimate this based on the current project status ("Status" column). Based on Google searches, the zoning and permitting stage takes about 0.5-1.5 months, and the actual construction takes about 1-3 additional years. Therefore, for datacenters in the zoning & permitting phase, we assume it'll take 1 year for zoning & permitting + 3 years for construction, i.e. expected operational date of 4 years from today. For data centers that are already under construction, we assume they will become operational in 2 years. There is one data center that has cleared the zoning and permitting stage but construction has not yet begun. Its status is marked as 'pre-construction'. For that one, we will take the upper limit of the construction time range (3 years). Also, for datacenters that are supposed to become operational in 2026, we'll assume that they'll become operational in December 2026 (end of the year). Otherwise, we'll take the first month of each year when the expected operational year is known.  

Estimating a datacenter's expected operational date based on its status and current year

In [58]:
#creating a new column for storing the expected operational month
upcoming_datacenters["Expected_Operational_Month"] = pd.NaT

#if Expected_Operational_Year = 2026, expected operational month assumed to be december 2026
mask = upcoming_datacenters["Expected_Operational_Year"] == "2026"
upcoming_datacenters.loc[mask, "Expected_Operational_Month"] = pd.to_datetime(
    "Dec " + upcoming_datacenters.loc[mask, "Expected_Operational_Year"].astype(str),
    format="%b %Y")

#if Expected_Operational_Year is known and is >2026, operational month assumed to be Jan of that year
mask = (upcoming_datacenters["Expected_Operational_Year"] != "2026") & (upcoming_datacenters.Expected_Operational_Year!="Not specified")
upcoming_datacenters.loc[mask, "Expected_Operational_Month"] = pd.to_datetime(
    "Jan " + upcoming_datacenters.loc[mask, "Expected_Operational_Year"].astype(str),
    format="%b %Y")

# If under construction, we assume operational month = July 2026 (present month) + 2 years = July 2028
mask = upcoming_datacenters["Status"] == "under construction"
upcoming_datacenters.loc[mask, "Expected_Operational_Month"] = pd.Timestamp("2028-07-01")
upcoming_datacenters

# If at the zoning & permitting stage, we assume operational month = July 2026 + 4 yrs = July 2030
mask = upcoming_datacenters["Status"] == "zoning and permitting"
upcoming_datacenters.loc[mask, "Expected_Operational_Month"] = pd.Timestamp("2030-07-01")
upcoming_datacenters

# If pre-construction, we assume operational month = July 2026 + 3 yrs = July 2029
mask = upcoming_datacenters["Status"] == "pre-construction"
upcoming_datacenters.loc[mask, "Expected_Operational_Month"] = pd.Timestamp("2029-07-01")


In [59]:
#checking results
upcoming_datacenters[[ "Expected_Operational_Year","Status", "Expected_Operational_Month"]]

,Expected_Operational_Year,Status,Expected_Operational_Month
0,2026,NaN,2026-12-01
1,Not specified,under construction,2028-07-01
2,Not specified,zoning and permitting,2030-07-01
3,Not specified,zoning and permitting,2030-07-01
4,2026,NaN,2026-12-01
5,2026,NaN,2026-12-01
6,2028,NaN,2028-01-01
7,2027,NaN,2027-01-01
8,Not specified,under construction,2028-07-01
9,Not specified,under construction,2028-07-01


In [60]:
#converting expected month column to datetime format
upcoming_datacenters["Expected_Operational_Month"] = pd.to_datetime(
    upcoming_datacenters.Expected_Operational_Month)
upcoming_datacenters.Expected_Operational_Month.dtype

dtype('<M8[ns]')

We can now calculating the expected number of data centers and total MW capacity for each month of each future year

In [61]:
#loading the current data so that we can add to it
current_datacenters = pd.read_csv("../Data/FormattedData/MonthlyFlow_and_AllFeatures.csv",
                              usecols=["Time", "datacenters_TotalMW", "datacenters_TotalNum"])

#extracting the most recent value 
most_recent = current_datacenters[current_datacenters["Time"]==current_datacenters.Time.max()]
most_recent

,Time,datacenters_TotalMW,datacenters_TotalNum
1468,2026-05-01,1212.0,13.0


In [62]:
#Range of dates of interest
months = pd.date_range(start='2026-06-01',
                        end=future_climate.Time.max(), freq='MS')

#Creating a an empty dataframe with the desired date range
future_datacenters_monthly = pd.DataFrame({'Time': months})

#Defining functions for calculating cumulative active operational MW capacity and number of active datacenters per month
def get_cumulative_load(date):
    active = upcoming_datacenters[upcoming_datacenters['Expected_Operational_Month'] <= date]
    return (active['Capacity_MW'].sum() + most_recent["datacenters_TotalMW"])

def get_cumulative_number(date):
    active = upcoming_datacenters[upcoming_datacenters['Expected_Operational_Month'] <= date]
    return (active['Capacity_MW'].count() + most_recent["datacenters_TotalNum"])

#Applying functions
future_datacenters_monthly['datacenters_TotalMW'] = future_datacenters_monthly['Time'].apply(get_cumulative_load)
future_datacenters_monthly['datacenters_TotalNum'] = future_datacenters_monthly['Time'].apply(get_cumulative_number)

future_datacenters_monthly.head()

,Time,datacenters_TotalMW,datacenters_TotalNum
0,2026-06-01,1212.0,13.0
1,2026-07-01,1212.0,13.0
2,2026-08-01,1212.0,13.0
3,2026-09-01,1212.0,13.0
4,2026-10-01,1212.0,13.0


### Combining all data into a single file

combining flow data with feature data (climate, population, irrigated acres, and data centers)

In [63]:
from functools import reduce

#gathering all tables into an iteratable array
dfs= [future_climate,
    futurepopulation_monthly,
    future_irrigated_acres,
    future_datacenters_monthly]

# Standardizing all time representations to a uniform string format (YYYY-MM-DD)
for df in dfs:
    df["Time"] = pd.to_datetime(df["Time"]).dt.strftime("%Y-%m-%d")


#merging all tables together using an Outer Join
all_merged = reduce(
    lambda left, right: pd.merge(left, right, on="Time", how="outer"), dfs
)

#Sorting chronologically from present to future
all_merged = all_merged.sort_values("Time").reset_index(drop=True)
all_merged.head()

,Time,Temp_Salt_SSP245,Temp_Verde_SSP245,Temp_Maricopa_SSP245,Precip_Salt_SSP585,Precip_Verde_SSP585,Precip_Maricopa_SSP585,Population,IrrigatedLand_Acres,datacenters_TotalMW,datacenters_TotalNum
0,2026-06-01,20.972582,21.974016,31.236763,6192.295867,3543.703456,834.732363,4.920049e+06,204522.0,1212.0,13.0
1,2026-07-01,25.034290,27.007235,36.358704,47477.990319,31365.076755,27081.072271,4.927591e+06,204522.0,1212.0,13.0
2,2026-08-01,20.949894,22.427137,32.220314,39420.638549,32705.906363,25530.815069,4.935385e+06,204522.0,1212.0,13.0
3,2026-09-01,18.813648,20.112804,30.477566,20961.371964,35531.386504,20287.838891,4.943178e+06,204522.0,1212.0,13.0
4,2026-10-01,15.049019,16.018923,25.397923,3136.122729,2153.111518,213.116947,4.950720e+06,204522.0,1212.0,13.0


Creating seperating dataframes for the two climate change scenarios

In [71]:
#removing SSP585 data to create a dataframe with only SSP245 climate data (and all other features)
ssp_245 = all_merged.drop(all_merged.filter(regex="_SSP585$").columns, axis=1)
ssp_245.rename(columns = lambda x: re.sub("_SSP245", "", x), inplace=True)

#removing SSP245 data to create a dataframe with only SSP585 climate data (and all other features)
ssp_585 = all_merged.drop(all_merged.filter(regex="_SSP245$").columns, axis=1)
ssp_585.rename(columns = lambda x: re.sub("_SSP585", "", x), inplace=True)
ssp_585.head()

,Time,Precip_Salt,Precip_Verde,Precip_Maricopa,Population,IrrigatedLand_Acres,datacenters_TotalMW,datacenters_TotalNum
0,2026-06-01,6192.295867,3543.703456,834.732363,4.920049e+06,204522.0,1212.0,13.0
1,2026-07-01,47477.990319,31365.076755,27081.072271,4.927591e+06,204522.0,1212.0,13.0
2,2026-08-01,39420.638549,32705.906363,25530.815069,4.935385e+06,204522.0,1212.0,13.0
3,2026-09-01,20961.371964,35531.386504,20287.838891,4.943178e+06,204522.0,1212.0,13.0
4,2026-10-01,3136.122729,2153.111518,213.116947,4.950720e+06,204522.0,1212.0,13.0


In [72]:
ssp_245.head()

,Time,Temp_Salt,Temp_Verde,Temp_Maricopa,Population,IrrigatedLand_Acres,datacenters_TotalMW,datacenters_TotalNum
0,2026-06-01,20.972582,21.974016,31.236763,4.920049e+06,204522.0,1212.0,13.0
1,2026-07-01,25.034290,27.007235,36.358704,4.927591e+06,204522.0,1212.0,13.0
2,2026-08-01,20.949894,22.427137,32.220314,4.935385e+06,204522.0,1212.0,13.0
3,2026-09-01,18.813648,20.112804,30.477566,4.943178e+06,204522.0,1212.0,13.0
4,2026-10-01,15.049019,16.018923,25.397923,4.950720e+06,204522.0,1212.0,13.0


Exporting to csv

In [ ]:
ssp_245.to_csv("../Data/FormattedData/FutureValues_SSP245_Climate_and_otherFeatures.csv", index= False)
ssp_585.to_csv("../Data/FormattedData/FutureValues_SSP585_Climate_and_otherFeatures.csv", index= False)